In [23]:
import numpy as np
import pandas as pd
import time
import datetime
import gc
import random
from nltk.corpus import stopwords
import re

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler,random_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import transformers
from transformers import BertForSequenceClassification, AdamW, BertConfig,BertTokenizer,get_linear_schedule_with_warmup
# Load model directly

In [30]:

from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained("microsoft/MiniLM-L12-H384-uncased")
tokenizer = AutoTokenizer.from_pretrained('microsoft/MiniLM-L12-H384-uncased', do_lower_case=True)
tokenizer.model_max_length = 512

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [32]:
train_df = pd.read_csv("../../comet-commonsense/train_english_atomic.csv")

In [33]:
train_df.head()

,utterances,label,speaker,oEffect,oReact,oWant,xAttr,xEffect,xIntent,xNeed,xReact,xWant,input
0,So you mentioned before that when you wake up ...,10,T_134,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,{'event': 'So you mentioned before that when y...,So you mentioned before that when you wake up ...
1,"I'm gonna have to, you know, hear what my cowo...",6,P_134,"{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","{'event': ""I'm gonna have to, you know, hear w...","I'm gonna have to, you know, hear what my cowo..."
2,So looking into one I'm going to make a mistak...,10,T_134,"{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...","{'event': ""So looking into one I'm going to ma...",So looking into one I'm going to make a mistak...
3,"I feel more anxious. You know, I think about l...",6,P_134,"{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","{'event': 'I feel more anxious. You know, I th...","I feel more anxious. You know, I think about l..."
4,So there's emotional consequences. And then th...,10,T_134,"{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...","{'event': ""So there's emotional consequences. ...",So there's emotional consequences. And then th...


In [34]:
print(' Original: ', train_df.iloc[0]["utterances"])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(train_df.iloc[0]["utterances"]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(train_df.iloc[0]["utterances"])))

 Original:  So you mentioned before that when you wake up in the morning, you you don't want to go to work, what's going through your mind?
Tokenized:  ['so', 'you', 'mentioned', 'before', 'that', 'when', 'you', 'wake', 'up', 'in', 'the', 'morning', ',', 'you', 'you', 'don', "'", 't', 'want', 'to', 'go', 'to', 'work', ',', 'what', "'", 's', 'going', 'through', 'your', 'mind', '?']
Token IDs:  [2061, 2017, 3855, 2077, 2008, 2043, 2017, 5256, 2039, 1999, 1996, 2851, 1010, 2017, 2017, 2123, 1005, 1056, 2215, 2000, 2175, 2000, 2147, 1010, 2054, 1005, 1055, 2183, 2083, 2115, 2568, 1029]


In [35]:
max_len = 0

# For every sentence...
for index,row in train_df.iterrows():
    sent = row["utterances"]
    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(sent, add_special_tokens=True)

    # Update the maximum sentence length.
    max_len = max(max_len, len(input_ids))

print('Max sentence length: ', max_len)

Token indices sequence length is longer than the specified maximum sequence length for this model (806 > 512). Running this sequence through the model will result in indexing errors


Max sentence length:  912


In [36]:
labels = train_df.label.values

In [37]:
input_ids = []
attention_masks = []

# For every tweet...
for index,sent in train_df.iterrows():
    tweet = row["utterances"]
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        tweet,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        max_length = max_len,           # Pad & truncate all sentences.
                        pad_to_max_length = True,
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                   )
    
    # Add the encoded sentence to the list.    
    input_ids.append(encoded_dict['input_ids'])
    
    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/home/kushagra20075/.local/lib/python3.6/site-packages/transformers/tokenization_utils_base.py:2269: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,


In [38]:
# Print sentence 0, now as a list of IDs.
print('Original: ', train_df.iloc[0]["utterances"])
print('Token IDs:', input_ids[0])

Original:  So you mentioned before that when you wake up in the morning, you you don't want to go to work, what's going through your mind?
Token IDs: tensor([ 101, 9061, 9061,  102,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0, 

In [39]:
# Combine the training inputs into a TensorDataset.
dataset = TensorDataset(input_ids, attention_masks, labels)

# Create a 90-10 train-validation split.

# Calculate the number of samples to include in each set.
train_size = int(0.8 * len(dataset))
#val_size = int(0.2 * len(dataset))
val_size = len(dataset)  - train_size

# Divide the dataset by randomly selecting samples.
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print('{:>5,} training samples'.format(train_size))
print('{:>5,} validation samples'.format(val_size))

6,581 training samples
1,646 validation samples


In [40]:
# The DataLoader needs to know our batch size for training, so we specify it 
# here. For fine-tuning BERT on a specific task, the authors recommend a batch 
# size of 16 or 32.
batch_size = 32

# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order. 
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

# For validation the order doesn't matter, so we'll just read them sequentially.
validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = SequentialSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

In [41]:
model = model.to(device)
optimizer = AdamW(model.parameters(),
                  lr = 2e-5, # args.learning_rate - default is 5e-5, our notebook had 2e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                )

In [42]:
# Number of training epochs. The BERT authors recommend between 2 and 4. 
# We chose to run for 4, but we'll see later that this may be over-fitting the
# training data.
epochs = 4

# Total number of training steps is [number of batches] x [number of epochs]. 
# (Note that this is not the same as the number of training samples).
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [43]:
# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

In [44]:
def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [45]:
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
training_stats = []

# Measure the total training time for the whole run.
total_t0 = time.time()

# For each epoch...
for epoch_i in range(0, epochs):
    
    # ========================================
    #               Training
    # ========================================
    # Perform one full pass over the training set.
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')
    # Measure how long the training epoch takes.
    t0 = time.time()
    total_train_loss = 0
    model.train()
    for step, batch in enumerate(train_dataloader):
        # Unpack this training batch from our dataloader. 
        #
        # As we unpack the batch, we'll also copy each tensor to the device using the 
        # `to` method.
        #
        # `batch` contains three pytorch tensors:
        #   [0]: input ids 
        #   [1]: attention masks
        #   [2]: labels 
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        optimizer.zero_grad()
        output = model(b_input_ids, 
                             token_type_ids=None, 
                             attention_mask=b_input_mask, 
                             labels=b_labels)        
        loss = output.loss
        total_train_loss += loss.item()
        # Perform a backward pass to calculate the gradients.
        loss.backward()
        # Clip the norm of the gradients to 1.0.
        # This is to help prevent the "exploding gradients" problem.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        # Update parameters and take a step using the computed gradient.
        # The optimizer dictates the "update rule"--how the parameters are
        # modified based on their gradients, the learning rate, etc.
        optimizer.step()
        # Update the learning rate.
        scheduler.step()

    # Calculate the average loss over all of the batches.
    avg_train_loss = total_train_loss / len(train_dataloader)            
    
    # Measure how long this epoch took.
    training_time = format_time(time.time() - t0)
    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epcoh took: {:}".format(training_time))
    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set.
    print("")
    print("Running Validation...")
    t0 = time.time()
    # Put the model in evaluation mode--the dropout layers behave differently
    # during evaluation.
    model.eval()
    # Tracking variables 
    total_eval_accuracy = 0
    best_eval_accuracy = 0
    total_eval_loss = 0
    nb_eval_steps = 0
    # Evaluate data for one epoch
    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        # Tell pytorch not to bother with constructing the compute graph during
        # the forward pass, since this is only needed for backprop (training).
        with torch.no_grad():        
            output= model(b_input_ids, 
                                   token_type_ids=None, 
                                   attention_mask=b_input_mask,
                                   labels=b_labels)
        loss = output.loss
        total_eval_loss += loss.item()
        # Move logits and labels to CPU if we are using GPU
        logits = output.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        # Calculate the accuracy for this batch of test sentences, and
        # accumulate it over all batches.
        total_eval_accuracy += flat_accuracy(logits, label_ids)
    # Report the final accuracy for this validation run.
    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    print("  Accuracy: {0:.2f}".format(avg_val_accuracy))
    # Calculate the average loss over all of the batches.
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    # Measure how long the validation run took.
    validation_time = format_time(time.time() - t0)
    if avg_val_accuracy > best_eval_accuracy:
        torch.save(model, 'bert_model')
        best_eval_accuracy = avg_val_accuracy
    #print("  Validation Loss: {0:.2f}".format(avg_val_loss))
    #print("  Validation took: {:}".format(validation_time))
    # Record all statistics from this epoch.
    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )
print("")
print("Training complete!")

print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))


======== Epoch 1 / 4 ========
Training...


RuntimeError: The expanded size of the tensor (912) must match the existing size (512) at non-singleton dimension 1.  Target sizes: [32, 912].  Tensor sizes: [1, 512]